# LeetCode 2002 — Maximum Product of the Length of Two Palindromic Subsequences bằng Python

Link đề: https://leetcode.com/problems/maximum-product-of-the-length-of-two-palindromic-subsequences/ · Độ khó: **Medium** · Chủ đề: String, Dynamic Programming, Backtracking, Bit Manipulation, Bitmask

Skeleton LeetCode:

```python
class Solution:
    def maxProduct(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Biểu diễn một **dãy con** bằng **bitmask** `n` bit và nhận ra `n ≤ 12` ⇒ chỉ `2¹² = 4096` dãy con: vét cạn được.
2. Thành thạo các phép bit: kiểm tra bit `mask >> i & 1`, giao `a & b`, phần bù `full ^ mask`, đếm bit `bin(m).count("1")`, và **duyệt mọi tập con** của một mask `sub = (sub - 1) & mask`.
3. Vét cạn ba trạng thái mỗi ký tự (vào A / vào B / bỏ) — `3ⁿ` — và hiểu vì sao cách duyệt tập con cũng có đúng `3ⁿ` cặp.
4. Tối ưu bằng **DP trên tập con**: `best[mask]` = độ dài palindrome dài nhất nằm trong `mask`, tính trong `O(2ⁿ · n)`.
5. Kết hợp với **LPS** (longest palindromic subsequence) để có thêm một góc nhìn, và đối chiếu mọi cách trên dữ liệu ngẫu nhiên.

## 2. Đề bài & ví dụ

Cho chuỗi `s`. Hãy chọn **hai dãy con palindrome rời nhau** (không cùng lấy ký tự ở một **chỉ số** nào) sao cho **tích độ dài** của chúng lớn nhất. Trả về tích lớn nhất đó.

- **Dãy con**: xóa đi một số (hoặc không) ký tự, giữ nguyên thứ tự các ký tự còn lại.
- **Palindrome**: đọc xuôi và ngược như nhau.

**Ví dụ 1:** `s = "leetcodecom"` → `9`. Chọn `"ete"` và `"cdc"`: `3 · 3 = 9`.

**Ví dụ 2:** `s = "bb"` → `1`. Mỗi dãy con lấy một chữ `b` (ở hai chỉ số khác nhau): `1 · 1 = 1`.

**Ví dụ 3:** `s = "accbcaxxcxx"` → `25`. Chọn `"accca"` và `"xxcxx"`: `5 · 5 = 25`.

**Ràng buộc:** `2 ≤ len(s) ≤ 12`, `s` chỉ gồm chữ thường tiếng Anh.

In [1]:
import random
import time
from functools import lru_cache
from itertools import product

examples = [("leetcodecom", 9), ("bb", 1), ("accbcaxxcxx", 25)]
for s, exp in examples:
    print(f"s={s!r} (n={len(s)}) -> kỳ vọng {exp}")

s='leetcodecom' (n=11) -> kỳ vọng 9
s='bb' (n=2) -> kỳ vọng 1
s='accbcaxxcxx' (n=11) -> kỳ vọng 25


## 3. Kiến thức nền

### 3.1 Bitmask biểu diễn dãy con

Với chuỗi độ dài `n`, mỗi số `mask` trong `[0, 2ⁿ)` ứng với một dãy con: bit `i` bằng 1 ⇔ lấy ký tự `s[i]`. Kiểm tra bit: `(mask >> i) & 1`.

In [2]:
def subseq(s, mask):
    return "".join(s[i] for i in range(len(s)) if (mask >> i) & 1)

s = "abcba"
for mask in [0b00000, 0b10001, 0b11011, 0b11111, 0b00110]:
    print(f"mask={mask:05b} (bit 0 ở bên phải) -> {subseq(s, mask)!r}")

mask=00000 (bit 0 ở bên phải) -> ''
mask=10001 (bit 0 ở bên phải) -> 'aa'
mask=11011 (bit 0 ở bên phải) -> 'abba'
mask=11111 (bit 0 ở bên phải) -> 'abcba'
mask=00110 (bit 0 ở bên phải) -> 'bc'


### 3.2 Các phép bit cần dùng

| Phép | Ý nghĩa |
|---|---|
| `a & b == 0` | hai dãy con **rời nhau** |
| `full = (1 << n) - 1` | mask lấy hết |
| `full ^ m` | **phần bù**: các chỉ số chưa dùng |
| `bin(m).count("1")` | số ký tự trong dãy con (Python ≥ 3.10 có `m.bit_count()`) |
| `m & (m - 1)` | xóa bit 1 thấp nhất |

In [3]:
n = 5; full = (1 << n) - 1
a, b = 0b10001, 0b01010
print("rời nhau?", a & b == 0, "| phần bù của a:", f"{full ^ a:05b}", "| số bit của a:", bin(a).count("1"))
print("xóa bit thấp nhất của 0b10110:", f"{0b10110 & (0b10110 - 1):05b}")

rời nhau? True | phần bù của a: 01110 | số bit của a: 2
xóa bit thấp nhất của 0b10110: 10100


### 3.3 Duyệt mọi tập con của một mask

Mẹo kinh điển: `sub = mask; while True: ...; if sub == 0: break; sub = (sub - 1) & mask`. Phép `(sub − 1) & mask` cho tập con **kế tiếp nhỏ hơn**. Tổng số cặp `(mask, sub ⊆ mask)` trên mọi mask là `3ⁿ` (mỗi phần tử: không trong mask / trong mask mà không trong sub / trong cả hai).

In [4]:
def submasks(mask):
    res, sub = [], mask
    while True:
        res.append(sub)
        if sub == 0:
            break
        sub = (sub - 1) & mask
    return res

print([f"{x:04b}" for x in submasks(0b1011)])
n = 6
print("tổng số cặp (mask, sub):", sum(len(submasks(m)) for m in range(1 << n)), "= 3^6 =", 3 ** 6)

['1011', '1010', '1001', '1000', '0011', '0010', '0001', '0000']
tổng số cặp (mask, sub): 729 = 3^6 = 729


### 3.4 Kiểm tra palindrome và LPS

`t == t[::-1]` kiểm tra palindrome. **LPS** (dãy con palindrome dài nhất) tính bằng DP khoảng: `L[i][j] = L[i+1][j-1] + 2` nếu `t[i] == t[j]`, ngược lại `max(L[i+1][j], L[i][j-1])`.

In [5]:
def lps(t):
    m = len(t)
    if m == 0:
        return 0
    L = [[0] * m for _ in range(m)]
    for i in range(m - 1, -1, -1):
        L[i][i] = 1
        for j in range(i + 1, m):
            if t[i] == t[j]:
                L[i][j] = L[i + 1][j - 1] + 2
            else:
                L[i][j] = max(L[i + 1][j], L[i][j - 1])
    return L[0][m - 1]

print("racecar" == "racecar"[::-1], "| LPS('bbbab') =", lps("bbbab"), "| LPS('leetcodecom') =", lps("leetcodecom"))

True | LPS('bbbab') = 4 | LPS('leetcodecom') = 3


## 4. Ý tưởng nền tảng / trực giác

Mỗi ký tự có **3 số phận**: vào dãy A, vào dãy B, hoặc bị bỏ. Ta muốn A, B đều palindrome và `|A|·|B|` lớn nhất.

Tính tay `s = "abab"` (n = 4):
- A = `"aba"` (chỉ số 0,1,2) thì còn lại chỉ `"b"` (chỉ số 3) ⇒ `3·1 = 3`.
- A = `"aa"` (0,2), B = `"bb"` (1,3) ⇒ `2·2 = 4`.
- A = `"bab"` (1,2,3), B = `"a"` (0) ⇒ `3`.

Vậy đáp án `4`. Chú ý: palindrome **dài nhất** (`"aba"`) **không** nằm trong lời giải tối ưu — chọn tham lam sẽ sai.

Vì `n ≤ 12`, số dãy con chỉ 4096. Ta có thể liệt kê hết các dãy con palindrome, rồi tìm cặp rời nhau tốt nhất.

In [6]:
s = "abab"; n = len(s)
pal_masks = [m for m in range(1, 1 << n) if subseq(s, m) == subseq(s, m)[::-1]]
print("Các mask palindrome của 'abab':")
for m in pal_masks:
    print(f"  {m:04b} -> {subseq(s, m)!r}")

Các mask palindrome của 'abab':
  0001 -> 'a'
  0010 -> 'b'
  0100 -> 'a'
  0101 -> 'aa'
  0111 -> 'aba'
  1000 -> 'b'
  1010 -> 'bb'
  1110 -> 'bab'


## 5. Cách 1 — Brute Force / baseline: gán 3 trạng thái cho từng ký tự

**Ý tưởng:** duyệt mọi phép gán `{A, B, bỏ}ⁿ` bằng `itertools.product`, dựng hai chuỗi, kiểm palindrome, lấy max tích.

In [7]:
def max_product_3n(s):
    best = 0
    for assign in product((0, 1, 2), repeat=len(s)):   # 0 = bỏ, 1 = A, 2 = B
        A = "".join(c for c, t in zip(s, assign) if t == 1)
        B = "".join(c for c, t in zip(s, assign) if t == 2)
        if A and B and A == A[::-1] and B == B[::-1]:
            best = max(best, len(A) * len(B))
    return best

for s, exp in examples[1:]:
    got = max_product_3n(s); print(s, "->", got, "| kỳ vọng", exp); assert got == exp
t = time.perf_counter(); r = max_product_3n("leetcode"); print("n=8:", r, f"({time.perf_counter() - t:.2f}s)")

bb -> 1 | kỳ vọng 1


accbcaxxcxx -> 25 | kỳ vọng 25
n=8: 3 (0.01s)


**Độ phức tạp:** `3ⁿ` phép gán, mỗi phép `O(n)` ⇒ `O(3ⁿ · n)`. Với `n = 12`: `531441 · 12 ≈ 6,4·10⁶` thao tác Python — chạy được (vài giây) nhưng chậm. Space `O(n)`.

**Nút thắt:** kiểm tra palindrome cho cùng một dãy con **lặp lại rất nhiều lần** (mỗi dãy A xuất hiện với mọi B). Ta nên **tính trước** "mask nào là palindrome" một lần.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — tiền xử lý:** tính `plen[mask]` = độ dài nếu `mask` là palindrome, `0` nếu không. Chỉ `2ⁿ · n` thao tác.

**Insight 2 — chỉ cần một dãy là palindrome "đang xét", dãy kia là "tốt nhất trong phần bù":** với mỗi mask palindrome `A`, đối tác tốt nhất là palindrome dài nhất **nằm trong** `full ^ A`. Gọi `best[M]` = max `plen[sub]` với `sub ⊆ M`. Khi đó:

$$\text{đáp án} = \max_{A:\ plen[A] > 0} plen[A] \cdot best[\text{full} \oplus A].$$

**Insight 3 — tính `best` bằng DP trên tập con:** `best[M] = max(plen[M], max_i best[M \ {i}])` (bỏ đi một bit bất kỳ). Thứ tự tăng dần của `M` đảm bảo `M \ {i} < M` đã tính.

**Quy tắc "nếu… thì…":**
- **Nếu** `A & B != 0` **thì** không hợp lệ (dùng chung chỉ số).
- **Nếu** `A` cố định **thì** chỉ cần biết palindrome dài nhất trong phần bù, không cần biết nó là gì.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tiền xử lý palindrome + duyệt cặp (mask, tập con của phần bù)

**Giải thích từng đoạn:**
1. `plen[mask]` cho mọi mask.
2. Với mỗi `A` palindrome, duyệt mọi `B ⊆ full ^ A` (mẹo `(sub-1) & comp`), lấy `plen[A] * plen[B]`.

Tổng số cặp ≤ `3ⁿ`, mỗi cặp `O(1)`.

In [8]:
def pal_lengths(s):
    n = len(s)
    plen = [0] * (1 << n)
    for mask in range(1, 1 << n):
        t = subseq(s, mask)
        if t == t[::-1]:
            plen[mask] = len(t)
    return plen

def max_product_pairs(s):
    n = len(s); full = (1 << n) - 1
    plen = pal_lengths(s)
    best = 0
    for A in range(1, 1 << n):
        if not plen[A]:
            continue
        comp = full ^ A
        sub = comp
        while sub:
            if plen[sub]:
                best = max(best, plen[A] * plen[sub])
            sub = (sub - 1) & comp
    return best

for s, exp in examples:
    got = max_product_pairs(s); print(s, "->", got); assert got == exp

leetcodecom -> 9
bb -> 1
accbcaxxcxx -> 25


In [9]:
class SolutionPairs:
    def maxProduct(self, s: str) -> int:
        n = len(s); full = (1 << n) - 1
        plen = [0] * (1 << n)
        for mask in range(1, 1 << n):
            t = [s[i] for i in range(n) if mask >> i & 1]
            if t == t[::-1]:
                plen[mask] = len(t)
        ans = 0
        for A in range(1, 1 << n):
            if plen[A]:
                comp = full ^ A; sub = comp
                while sub:
                    if plen[sub]:
                        ans = max(ans, plen[A] * plen[sub])
                    sub = (sub - 1) & comp
        return ans

print([SolutionPairs().maxProduct(s) for s, _ in examples])

[9, 1, 25]


**Độ phức tạp Cách 2:** tiền xử lý `O(2ⁿ · n)`, duyệt cặp `O(3ⁿ)` worst case (thực tế ít hơn vì chỉ xét `A` palindrome). Space `O(2ⁿ)`.

### 7.2 Cách 3 — Mỗi mask palindrome + LPS của phần còn lại

Thay vì duyệt tập con của phần bù, ta tính **LPS** của chuỗi gồm các ký tự trong phần bù (DP khoảng `O(n²)`). Đúng vì palindrome dài nhất nằm trong phần bù chính là LPS của chuỗi con tạo bởi phần bù.

In [10]:
def max_product_lps(s):
    n = len(s); full = (1 << n) - 1
    plen = pal_lengths(s)
    best = 0
    for A in range(1, 1 << n):
        if plen[A]:
            rest = subseq(s, full ^ A)
            best = max(best, plen[A] * lps(rest))
    return best

for s, exp in examples:
    assert max_product_lps(s) == exp
print([max_product_lps(s) for s, _ in examples])

[9, 1, 25]


**Độ phức tạp Cách 3:** có tới `P` mask palindrome (`P ≤ 2ⁿ`), mỗi cái tốn LPS `O(n²)` ⇒ `O(2ⁿ · n²)` ≈ `4096 · 144 ≈ 6·10⁵`. Space `O(2ⁿ + n²)`.

### 7.3 Cách 4 — DP trên tập con `best[M]` (tối ưu nhất)

**Giải thích từng đoạn:**
1. `plen[mask]` như trên.
2. `best[M] = plen[M]`; với mỗi bit `i` trong `M`: `best[M] = max(best[M], best[M ^ (1<<i)])`.
3. Đáp án = `max(plen[A] * best[full ^ A])` trên các `A` palindrome.

In [11]:
def max_product_sos(s):
    n = len(s); full = (1 << n) - 1
    plen = pal_lengths(s)
    best = plen[:]                       # best[M] = palindrome dài nhất nằm trong M
    for M in range(1, 1 << n):
        for i in range(n):
            if M >> i & 1:
                v = best[M ^ (1 << i)]
                if v > best[M]:
                    best[M] = v
    return max(plen[A] * best[full ^ A] for A in range(1, 1 << n) if plen[A])

for s, exp in examples:
    assert max_product_sos(s) == exp
print([max_product_sos(s) for s, _ in examples])

[9, 1, 25]


In [12]:
class SolutionSOS:
    def maxProduct(self, s: str) -> int:
        n = len(s); full = (1 << n) - 1
        plen = [0] * (1 << n)
        for mask in range(1, 1 << n):
            t = [s[i] for i in range(n) if mask >> i & 1]
            if t == t[::-1]:
                plen[mask] = len(t)
        best = plen[:]
        for M in range(1, 1 << n):
            for i in range(n):
                if M >> i & 1 and best[M ^ (1 << i)] > best[M]:
                    best[M] = best[M ^ (1 << i)]
        return max(plen[A] * best[full ^ A] for A in range(1, 1 << n) if plen[A])

print([SolutionSOS().maxProduct(s) for s, _ in examples])

[9, 1, 25]


**Độ phức tạp Cách 4:** `O(2ⁿ · n)` cho cả tiền xử lý lẫn DP ⇒ với `n = 12` chỉ ~`5·10⁴` bước. Space `O(2ⁿ)`.

**Tối ưu nhỏ cho `plen`:** có thể tính palindrome của `mask` nhanh hơn bằng hai con trỏ trên các chỉ số bit, nhưng với `n ≤ 12` không cần.

## 8. TRACE từng bước

Trace Cách 4 trên `s = "abab"` (n = 4, bit 0 = `s[0]`):

1. Mask palindrome: `0001 'a'`, `0010 'b'`, `0100 'a'`, `1000 'b'`, `0101 'aa'`, `1010 'bb'`, `0111 'aba'`, `1110 'bab'`.
2. Với `A = 0101 ('aa')`: phần bù `1010` có `best = 2` (`'bb'`) ⇒ `2·2 = 4`.
3. Với `A = 0111 ('aba')`: phần bù `1000` có `best = 1` ⇒ `3`.

| A | chuỗi A | plen[A] | phần bù | best[phần bù] | tích |
|---|---|---|---|---|---|
| 0101 | aa | 2 | 1010 | 2 | **4** |
| 0111 | aba | 3 | 1000 | 1 | 3 |
| 1110 | bab | 3 | 0001 | 1 | 3 |

In [13]:
def trace_sos(s):
    n = len(s); full = (1 << n) - 1
    plen = pal_lengths(s)
    best = plen[:]
    for M in range(1, 1 << n):
        for i in range(n):
            if M >> i & 1:
                best[M] = max(best[M], best[M ^ (1 << i)])
    ans = 0
    for A in range(1, 1 << n):
        if plen[A]:
            comp = full ^ A
            prod = plen[A] * best[comp]
            ans = max(ans, prod)
            print(f"A={A:0{n}b} {subseq(s, A)!r:8s} plen={plen[A]} | bù={comp:0{n}b} "
                  f"{subseq(s, comp)!r:8s} best={best[comp]} -> tích={prod}, ans={ans}")
    return ans

print("Đáp án:", trace_sos("abab"))

A=0001 'a'      plen=1 | bù=1110 'bab'    best=3 -> tích=3, ans=3
A=0010 'b'      plen=1 | bù=1101 'aab'    best=2 -> tích=2, ans=3
A=0100 'a'      plen=1 | bù=1011 'abb'    best=2 -> tích=2, ans=3
A=0101 'aa'     plen=2 | bù=1010 'bb'     best=2 -> tích=4, ans=4
A=0111 'aba'    plen=3 | bù=1000 'b'      best=1 -> tích=3, ans=4
A=1000 'b'      plen=1 | bù=0111 'aba'    best=3 -> tích=3, ans=4
A=1010 'bb'     plen=2 | bù=0101 'aa'     best=2 -> tích=4, ans=4
A=1110 'bab'    plen=3 | bù=0001 'a'      best=1 -> tích=3, ans=4
Đáp án: 4


In [14]:
def trace_best_table(s):
    n = len(s)
    plen = pal_lengths(s); best = plen[:]
    for M in range(1, 1 << n):
        src = "chính nó" if plen[M] else "-"
        for i in range(n):
            if M >> i & 1 and best[M ^ (1 << i)] > best[M]:
                best[M] = best[M ^ (1 << i)]; src = f"bỏ bit {i}"
        print(f"M={M:0{n}b} {subseq(s, M)!r:7s} plen={plen[M]} best={best[M]} ({src})")

trace_best_table("aba")

M=001 'a'     plen=1 best=1 (chính nó)
M=010 'b'     plen=1 best=1 (chính nó)
M=011 'ab'    plen=0 best=1 (bỏ bit 0)
M=100 'a'     plen=1 best=1 (chính nó)
M=101 'aa'    plen=2 best=2 (chính nó)
M=110 'ba'    plen=0 best=1 (bỏ bit 1)
M=111 'aba'   plen=3 best=3 (chính nó)


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def maxProduct(self, s: str) -> int:
        n = len(s)
        full = (1 << n) - 1
        # plen[mask] = độ dài nếu dãy con mask là palindrome, ngược lại 0
        plen = [0] * (1 << n)
        for mask in range(1, 1 << n):
            t = [s[i] for i in range(n) if mask >> i & 1]
            if t == t[::-1]:
                plen[mask] = len(t)
        # best[M] = palindrome dài nhất là tập con của M
        best = plen[:]
        for M in range(1, 1 << n):
            for i in range(n):
                if M >> i & 1 and best[M ^ (1 << i)] > best[M]:
                    best[M] = best[M ^ (1 << i)]
        return max(plen[A] * best[full ^ A] for A in range(1, 1 << n) if plen[A])

for s, exp in examples:
    assert Solution().maxProduct(s) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Không gian lời giải:** một lời giải hợp lệ là cặp mask `(A, B)` với `A & B = 0`, `A, B ≠ 0`, cả hai palindrome. Giá trị `plen[A]·plen[B]`.

**`best` đúng (quy nạp theo số bit của `M`):** `best[M] = max_{sub ⊆ M} plen[sub]`. Với `M = 0` thì `best = 0`. Với `M` khác 0: mọi tập con `sub` của `M` hoặc bằng `M`, hoặc thiếu ít nhất một bit `i` của `M` nên `sub ⊆ M \ {i}`; theo giả thiết quy nạp `best[M \ {i}]` đã ≥ `plen[sub]`. Ngược lại mọi giá trị ta lấy max đều là `plen` của một tập con của `M`. Vậy công thức đúng. Thứ tự duyệt `M` tăng dần hợp lệ vì `M \ {i} < M`.

**Đáp án đúng:** với lời giải tối ưu `(A*, B*)`: `B* ⊆ full ^ A*` ⇒ `best[full ^ A*] ≥ plen[B*]` ⇒ giá trị ta tính ≥ tối ưu. Ngược lại mỗi giá trị `plen[A]·best[full^A]` (khi `best > 0`) ứng với một cặp hợp lệ thật: `A` và palindrome đạt `best` trong phần bù ⇒ ≤ tối ưu. Trường hợp phần bù không chứa palindrome nào (chỉ khi `A = full`) cho tích `0`, không ảnh hưởng `max` vì luôn tồn tại cặp hai ký tự đơn ở hai chỉ số khác nhau (`n ≥ 2`) cho tích `≥ 1`.

## 11. Phân tích Time / Space

Gọi `n = len(s) ≤ 12`, `N = 2ⁿ`.

| Cách | Time | Space | n = 12 (ước lượng) |
|---|---|---|---|
| 1. Gán 3 trạng thái | `O(3ⁿ · n)` | `O(n)` | ~6,4·10⁶ |
| 2. Cặp (mask, tập con phần bù) | `O(N·n + 3ⁿ)` | `O(N)` | ~5,8·10⁵ |
| 3. Mask + LPS phần bù | `O(N · n²)` | `O(N + n²)` | ~5,9·10⁵ |
| 4. DP tập con | `O(N · n)` | `O(N)` | ~4,9·10⁴ |

In [16]:
rng = random.Random(2002)
worst = ["".join(rng.choice("ab") for _ in range(12)), "a" * 12, "abcdefghijkl"]
for s in worst:
    row = []
    for name, fn in [("pairs", max_product_pairs), ("lps", max_product_lps), ("sos", max_product_sos)]:
        t = time.perf_counter(); r = fn(s); row.append(f"{name}={r} ({time.perf_counter() - t:.3f}s)")
    print(f"{s}: " + ", ".join(row))

ababbaabbaab: pairs=36 (0.012s), lps=36 (0.006s), sos=36 (0.006s)
aaaaaaaaaaaa: pairs=36 (0.033s), lps=36 (0.013s), sos=36 (0.006s)
abcdefghijkl: pairs=1 (0.004s), lps=1 (0.003s), sos=1 (0.006s)


## 12. Bộ test

- Ví dụ đề (cả ba cách tối ưu + lớp nộp).
- Biên: `n = 2` khác/giống chữ, toàn một chữ (`"a"*n` ⇒ `⌊n/2⌋·⌈n/2⌉`), mọi chữ khác nhau (⇒ `1`).
- Đối chiếu với brute force `3ⁿ` trên chuỗi ngẫu nhiên `n ≤ 8` (seed cố định), bảng chữ nhỏ để có nhiều palindrome.

In [17]:
fns = {
    "pairs": max_product_pairs, "lps": max_product_lps, "sos": max_product_sos,
    "Solution": lambda s: Solution().maxProduct(s),
    "SolutionPairs": lambda s: SolutionPairs().maxProduct(s),
    "SolutionSOS": lambda s: SolutionSOS().maxProduct(s),
}
for s, exp in examples:
    for name, fn in fns.items():
        assert fn(s) == exp, (name, s)
assert Solution().maxProduct("ab") == 1
assert Solution().maxProduct("aa") == 1
for n in range(2, 13):
    assert Solution().maxProduct("a" * n) == (n // 2) * ((n + 1) // 2)
assert Solution().maxProduct("abcdefghijkl") == 1
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [18]:
rng = random.Random(42)
for _ in range(250):
    n = rng.randint(2, 8)
    s = "".join(rng.choice(rng.choice(["ab", "abc", "abcd", "xyzw"])) for _ in range(n))
    want = max_product_3n(s)
    for name, fn in fns.items():
        assert fn(s) == want, (name, s)
print("250 test ngẫu nhiên: mọi cách khớp brute force 3^n")

250 test ngẫu nhiên: mọi cách khớp brute force 3^n


In [19]:
rng = random.Random(9)
for _ in range(6):
    s = "".join(rng.choice("abc") for _ in range(12))
    assert max_product_pairs(s) == max_product_lps(s) == max_product_sos(s) == Solution().maxProduct(s)
print("6 test n = 12: các cách tối ưu khớp nhau")

6 test n = 12: các cách tối ưu khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên điều kiện rời nhau

Nếu cho phép hai dãy con dùng chung chỉ số, với `"bb"` ta chọn `"bb"` hai lần ⇒ `4`.

In [20]:
def wrong_not_disjoint(s):
    n = len(s); plen = pal_lengths(s)
    return max(plen[a] * plen[b] for a in range(1, 1 << n) for b in range(1, 1 << n))  # SAI: không kiểm a & b

print("sai:", wrong_not_disjoint("bb"), "| đúng:", Solution().maxProduct("bb"))

sai: 4 | đúng: 1


In [21]:
def fixed_disjoint(s):
    n = len(s); plen = pal_lengths(s)
    return max(plen[a] * plen[b] for a in range(1, 1 << n) for b in range(1, 1 << n) if a & b == 0)

print("sửa:", fixed_disjoint("bb"))

sửa: 1


### Lỗi 2 — Tham lam: lấy LPS của cả chuỗi trước, rồi LPS phần còn lại

Palindrome dài nhất không nhất thiết thuộc lời giải tối ưu. Ta tìm một phản ví dụ bằng cách thử mọi chuỗi ngắn trên `{a, b}`.

In [22]:
def greedy_lps_first(s):
    n = len(s); full = (1 << n) - 1
    plen = pal_lengths(s)
    # một palindrome dài nhất, nhưng chừa lại ít nhất một ký tự cho dãy thứ hai
    A = max(range(1, full), key=lambda m: plen[m])
    return plen[A] * lps(subseq(s, full ^ A))

counter = next(("".join(t) for L in range(2, 9) for t in product("ab", repeat=L)
                if greedy_lps_first("".join(t)) != Solution().maxProduct("".join(t))))
print("phản ví dụ:", counter, "| tham lam:", greedy_lps_first(counter), "| đúng:", Solution().maxProduct(counter))

phản ví dụ: aaaa | tham lam: 3 | đúng: 4


In [23]:
print("sửa: xét MỌI palindrome A ->", max_product_lps(counter))

sửa: xét MỌI palindrome A -> 4


### Lỗi 3 — Nhầm dãy con (subsequence) với xâu con liên tiếp (substring)

Nếu chỉ xét các **đoạn liên tiếp** palindrome, ví dụ 3 `"accbcaxxcxx"` không lấy được `"accca"`.

In [24]:
def wrong_substring(s):
    n = len(s); best = 0
    subs = [(i, j) for i in range(n) for j in range(i + 1, n + 1) if s[i:j] == s[i:j][::-1]]
    for (a, b) in subs:
        for (c, d) in subs:
            if b <= c:                        # hai đoạn không chồng nhau
                best = max(best, (b - a) * (d - c))
    return best

print("sai:", wrong_substring("accbcaxxcxx"), "| đúng:", Solution().maxProduct("accbcaxxcxx"))

sai: 15 | đúng: 25


In [25]:
print("sửa (dãy con qua bitmask):", max_product_sos("accbcaxxcxx"))

sửa (dãy con qua bitmask): 25


### Lỗi 4 — Vòng duyệt tập con không dừng

Viết `while True: ...; sub = (sub - 1) & comp` mà **không** kiểm `sub == 0` thì `(0 − 1) & comp = comp`, vòng lặp quay lại từ đầu mãi mãi. Ta demo với "cầu chì" giới hạn số vòng.

In [26]:
def wrong_submask_loop(comp, cap=50):
    sub, seen = comp, []
    for _ in range(cap):                 # đáng lẽ là while True
        seen.append(sub)
        sub = (sub - 1) & comp           # SAI: không dừng khi sub == 0
    return seen

r = wrong_submask_loop(0b101)
print("sai: sau 50 vòng vẫn lặp:", r[:10], "...")

sai: sau 50 vòng vẫn lặp: [5, 4, 1, 0, 5, 4, 1, 0, 5, 4] ...


In [27]:
print("sửa:", submasks(0b101))

sửa: [5, 4, 1, 0]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **`n ≤ 12–20`** + "chọn tập con" ⇒ nghĩ ngay **bitmask** `2ⁿ`.
- **Hai tập rời nhau** ⇒ duyệt `A` rồi tập con của **phần bù** (`3ⁿ`), hoặc DP `best[M]` trên tập con để còn `2ⁿ · n`.
- "Tốt nhất trong một tập con của M" ⇒ DP **bỏ từng bit** (`best[M] = max(best[M ^ bit])`) — một dạng SOS DP (sum/max over subsets).
- Có `max`/`tích` của hai phần độc lập ⇒ cố định một phần, tối ưu phần còn lại.
- Luôn có brute force `3ⁿ` làm chuẩn cho `n ≤ 8`.

## 15. Biến thể của bài

1. **Tổng độ dài lớn nhất** của hai dãy con palindrome rời nhau — cùng khung, đổi `*` thành `+`.
2. **Ba dãy con palindrome rời nhau** (bài tập 5) — duyệt `A`, rồi tập con của phần bù, rồi `best` của phần còn lại.
3. **LeetCode 1745 — Palindrome Partitioning IV** (chia thành 3 đoạn palindrome liên tiếp): substring thay vì subsequence.
4. **`n` lớn (tới 10⁵), hai xâu con palindrome không chồng nhau, tích lớn nhất** (LeetCode 1960, Hard): dùng Manacher thay cho bitmask.
5. **LeetCode 516 — Longest Palindromic Subsequence**: chính là hàm `lps` dùng ở Cách 3.

In [28]:
def max_sum_two(s):
    n = len(s); full = (1 << n) - 1
    plen = pal_lengths(s); best = plen[:]
    for M in range(1, 1 << n):
        for i in range(n):
            if M >> i & 1: best[M] = max(best[M], best[M ^ (1 << i)])
    return max(plen[A] + best[full ^ A] for A in range(1, 1 << n) if plen[A] and best[full ^ A])

def max_sum_brute(s):
    best = 0
    for assign in product((0, 1, 2), repeat=len(s)):
        A = "".join(c for c, t in zip(s, assign) if t == 1); B = "".join(c for c, t in zip(s, assign) if t == 2)
        if A and B and A == A[::-1] and B == B[::-1]: best = max(best, len(A) + len(B))
    return best

rng = random.Random(1)
for _ in range(80):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(2, 7)))
    assert max_sum_two(s) == max_sum_brute(s)
print("Biến thể tổng độ dài (tự dựng) khớp brute force; 'abab' ->", max_sum_two("abab"))

Biến thể tổng độ dài (tự dựng) khớp brute force; 'abab' -> 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Gán 3 trạng thái | `product((0,1,2), repeat=n)` | `O(3ⁿ·n)` | `O(n)` | chuẩn kiểm tra, `n ≤ 8` |
| 2. Cặp mask + tập con phần bù | tiền xử lý `plen`, duyệt `3ⁿ` cặp | `O(3ⁿ)` | `O(2ⁿ)` | dễ nghĩ, đủ nhanh |
| 3. Mask + LPS phần bù | DP khoảng cho phần bù | `O(2ⁿ·n²)` | `O(2ⁿ+n²)` | khi đã quen LPS |
| 4. DP tập con `best[M]` | max trên tập con bằng bỏ bit | `O(2ⁿ·n)` | `O(2ⁿ)` | **nộp bài**, nhanh nhất |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `maxProduct("aaaa")`.
2. **Tính tay:** `maxProduct("abcd")` và `maxProduct("abcba")`.
3. **Điền TODO:** hoàn thiện `is_pal_mask(s, mask)` **không dựng chuỗi**: dùng hai con trỏ trên danh sách chỉ số bit.
4. **Tự viết lại:** viết lại Cách 4 (DP tập con) không nhìn code.
5. **Mini challenge:** `max_product_three(s)` — tích lớn nhất của **ba** dãy con palindrome rời nhau (`n ≤ 10`). Kiểm bằng brute force `4ⁿ` với `n ≤ 7`.

In [29]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def is_pal_mask(s, mask):
    idx = [i for i in range(len(s)) if mask >> i & 1]
    # TODO: hai con trỏ l, r trên idx
    pass

In [30]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_max_product(s):
    # TODO: plen -> best -> max(plen[A] * best[full ^ A])
    pass

In [31]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def max_product_three(s):
    # TODO
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Mọi dãy con của `"aaaa"` đều palindrome. (2) Độ dài `a + b ≤ 4`. (3) Max `a·b` khi `a = b = 2`.

**Bài 2.** (1) `"abcd"` không có hai chữ giống nhau ⇒ mọi palindrome dài 1. (2) `"abcba"`: thử `"bcb"` + `"a"`, `"aca"`+`"b"`, `"aa"`+`"bb"`... (3) Có cặp nào cho `4` không? `"aba"` (0,1,4) + `"c"`? = 3; `"aa"` + `"bcb"` = 6!

**Bài 3.** (1) `l = 0, r = len(idx) - 1`. (2) So `s[idx[l]]` với `s[idx[r]]`. (3) Tiến `l`, lùi `r` tới khi gặp nhau.

**Bài 4.** (1) `best = plen[:]`. (2) Duyệt `M` tăng dần, thử bỏ từng bit. (3) Chỉ nhân với `A` palindrome.

**Bài 5.** (1) Tính `best[M]` như Cách 4. (2) Với mỗi `A` palindrome, duyệt `B` palindrome là tập con của phần bù. (3) Tích `plen[A]·plen[B]·best[phần bù của A|B]`.

## 19. Đáp án tham khảo

In [32]:
# Bài 1, 2
assert Solution().maxProduct("aaaa") == 4
assert Solution().maxProduct("abcd") == 1
assert Solution().maxProduct("abcba") == 6
print("Bài 1: 4 | Bài 2: abcd -> 1, abcba -> 6")

Bài 1: 4 | Bài 2: abcd -> 1, abcba -> 6


In [33]:
# Bài 3, 4
def is_pal_mask_ans(s, mask):
    idx = [i for i in range(len(s)) if mask >> i & 1]
    l, r = 0, len(idx) - 1
    while l < r:
        if s[idx[l]] != s[idx[r]]:
            return False
        l += 1; r -= 1
    return True

def my_max_product_ans(s):
    n = len(s); full = (1 << n) - 1
    plen = [bin(m).count("1") if m and is_pal_mask_ans(s, m) else 0 for m in range(1 << n)]
    best = plen[:]
    for M in range(1, 1 << n):
        for i in range(n):
            if M >> i & 1: best[M] = max(best[M], best[M ^ (1 << i)])
    return max(plen[A] * best[full ^ A] for A in range(1, 1 << n) if plen[A])

rng = random.Random(3)
for _ in range(100):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(2, 10)))
    assert all(is_pal_mask_ans(s, m) == (subseq(s, m) == subseq(s, m)[::-1]) for m in range(1 << len(s)))
    assert my_max_product_ans(s) == Solution().maxProduct(s)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [34]:
# Bài 5
def max_product_three_ans(s):
    n = len(s); full = (1 << n) - 1
    plen = pal_lengths(s); best = plen[:]
    for M in range(1, 1 << n):
        for i in range(n):
            if M >> i & 1: best[M] = max(best[M], best[M ^ (1 << i)])
    ans = 0
    for A in range(1, 1 << n):
        if not plen[A]: continue
        comp = full ^ A; B = comp
        while B:
            if plen[B]:
                ans = max(ans, plen[A] * plen[B] * best[comp ^ B])
            B = (B - 1) & comp
    return ans

def three_brute(s):
    best = 0
    for assign in product(range(4), repeat=len(s)):
        parts = ["".join(c for c, t in zip(s, assign) if t == k) for k in (1, 2, 3)]
        if all(p and p == p[::-1] for p in parts):
            best = max(best, len(parts[0]) * len(parts[1]) * len(parts[2]))
    return best

rng = random.Random(8)
for _ in range(25):
    s = "".join(rng.choice("ab") for _ in range(rng.randint(3, 7)))
    assert max_product_three_ans(s) == three_brute(s)
print("Bài 5: OK; 'aaaaaa' ->", max_product_three_ans("aaaaaa"), "| 'abcabcabc' ->", max_product_three_ans("abcabcabc"))

Bài 5: OK; 'aaaaaa' -> 8 | 'abcabcabc' -> 27


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "hai tập con rời nhau tốt nhất":**

```python
full = (1 << n) - 1
val = [score(mask) for mask in range(1 << n)]     # 0 nếu không hợp lệ
best = val[:]
for M in range(1, 1 << n):
    for i in range(n):
        if M >> i & 1:
            best[M] = max(best[M], best[M ^ (1 << i)])
ans = max(val[A] * best[full ^ A] for A in range(1, 1 << n) if val[A])
```

In [35]:
def template(s):
    n = len(s); full = (1 << n) - 1
    val = [0] + [bin(m).count("1") if is_pal_mask_ans(s, m) else 0 for m in range(1, 1 << n)]
    best = val[:]
    for M in range(1, 1 << n):
        for i in range(n):
            if M >> i & 1: best[M] = max(best[M], best[M ^ (1 << i)])
    return max(val[A] * best[full ^ A] for A in range(1, 1 << n) if val[A])
print([template(s) for s, _ in examples])

[9, 1, 25]


**Checklist:**

- [ ] Hai mask phải **rời nhau** (`A & B == 0`), dùng phần bù `full ^ A`.
- [ ] Dãy con (subsequence), **không** phải xâu con liên tiếp.
- [ ] Không tham lam lấy palindrome dài nhất trước.
- [ ] Vòng duyệt tập con dừng khi `sub == 0`.
- [ ] `best[M]` duyệt `M` tăng dần (tập con nhỏ hơn đã tính).
- [ ] Đã thử `"bb"`, `"ab"`, `"a"*12`, chuỗi toàn chữ khác nhau.